In [ ]:
import math
import numpy as np
import random
from numpy.random import uniform, exponential
import seaborn as sns
import time,timeit
from typing import Any, Callable, List, Optional, Tuple


def simulate(X,n_sim,ddof=0,**kwargs):
    """
    Parameters:
    -----------
    X : callable
        A function (or callable object) that generates a single simulated value when called with **kwargs.
    n_sim : int
        Number of simulations to run.
    ddof : int, optional (default=0)
        Delta degrees of freedom for variance and standard deviation calculations.
        If ddof=0 (default), computes population variance/std.
        If ddof=1, computes sample variance/std (unbiased estimator).
    **kwargs : dict
        Additional keyword arguments passed to the function X.

    Returns:
    --------
    s : ndarray
        Array of all simulated values (shape: (n_sim,)).
    mean_sim : float
        Mean of the simulated values.
    median_sim : float
        Median of the simulated values.
    var_sim : float
        Variance of the simulated values (with specified ddof).
    std_sim : float
        Standard deviation of the simulated values (with specified ddof).
    """
    s = np.empty(n_sim)
    for i in range(n_sim):
      s[i]=X(**kwargs)
    mean_sim = np.mean(s)
    median_sim = np.median(s)
    var_sim = np.var(s,ddof=ddof)
    std_sim = np.std(s,ddof=ddof)
    return s,mean_sim,median_sim,var_sim,std_sim


def montecarlo_ab_intervalo(func, a, b, max_sim, min_sim, threshold):

    media = func(random.random())
    varianza = 0
    n = 1
    intervalo = math.sqrt(varianza / n)

    while n <= min_sim or intervalo > threshold:
        sim = func(a + (b - a) * random.random()) * (b-a) #transformo la x
        media_ant = media
        media = media_ant + (sim - media_ant) / (n + 1)
        varianza_ant = varianza
        varianza = (1 - (1 / n)) * varianza + (n + 1) * (media - media_ant) ** 2
        n += 1
        intervalo = math.sqrt(varianza / n)
    return media, math.sqrt(varianza), intervalo, n

"""
#formula de las notas
zalpha_2 = 1.95 # alpha = 0.05
L = 2 * 0.001
d = L / (2 * zalpha_2)
"""

def montecarlo_0_inf_intervalo(func, max_sim, n_min, threshold):

    media = func(random.random())
    varianza = 0
    n = 1
    intervalo = math.sqrt(varianza / n)

    def h(u):
        return (1 / (u**2)) * func((1 / u) - 1)

    while n <= n_min or intervalo > threshold:
      sim=h(random.random())
      media_ant = media
      media = media_ant + (sim - media_ant) / (n + 1)
      varianza_ant = varianza
      varianza = (1 - (1 / n)) * varianza + (n + 1) * (media - media_ant) ** 2
      n += 1
      intervalo = math.sqrt(varianza / n)
    return media, math.sqrt(varianza), intervalo, n

Ejercicio 1: Genere n valores de una variable aleatoria normal estándar de manera tal que se cumplan
las condiciones: n ≥ 100 y S/
√
n < 0,1, siendo S el estimador de la desviación estándar de los n datos
generados.

a) ¿Cuál es el número de datos generados efectivamente?

b) ¿Cuál es la media muestral de los datos generados?

c) ¿Cuál es la varianza muestral de los datos generados?


In [ ]:
def exponencial(lamda: float = 1.0) -> float:
    u = 1 - random.random()
    return -math.log(u)/lamda

#Generador de normales por rechazo de las notas
def normal_rechazo(mu: float = 0.0, sigma: float = 1.0) -> float:
    while True:
        Y1 = exponencial()
        Y2 = exponencial()
        if Y2 >= ((Y1 - 1) ** 2)/2:
            if random.random() <= 1/2:
                return Y1 * sigma + mu
            else:
                return Y2 * sigma + mu

def ejercicio1(cant_valores: int = 100) -> list[float]:
    valores = []

    while True:
        #Genero las normales y guardo todos los valores en un arreglo (al menos 100 de entrada).
        valores_insertar = [normal_rechazo() for _ in range(cant_valores - len(valores))]
        valores.extend(valores_insertar)

        #Actualizo la varianza y la media
        mean = sum(valores)/len(valores)
        mean_sq = sum(valores ** 2 for valores in valores) / len(valores)
        variance = mean_sq - (mean**2)
        deviance = math.sqrt(variance)

        #Salgo cuando cumpla
        if deviance/math.sqrt(len(valores)) <= 0.1:
            break
        else:
            cant_valores += 1 #100 + la cantidad de veces que no cumpla
    return valores, mean, variance


contador: int = 0
media: float = 0
varianza: float = 0
n_sim = 10000

for _ in range(n_sim):

  values, mean, variance = ejercicio1()
  contador += len(values)
  media += mean
  varianza += variance

contador /= n_sim
media /= n_sim
varianza /= n_sim

print("a) ¿Cuál es el número de datos generados efectivamente?")
print(f"{contador}")
print("b) ¿Cuál es la media muestral de los datos generados?")
print(f"{media}")
print("c) ¿Cuál es la varianza muestral de los datos generados?")
print(f"{varianza}")

a) ¿Cuál es el número de datos generados efectivamente?
101.6618
b) ¿Cuál es la media muestral de los datos generados?
0.9977523297439976
c) ¿Cuál es la varianza muestral de los datos generados?
0.7377358305749029


Ejercicio 2)

In [ ]:
def fun_i(x: float) -> float:
    return math.exp(x) / math.sqrt(2 * x)

def montecarlo_0_1(func: Callable, d_threshold: float):

    media = func(uniform())
    varianza, n = 0, 1
    while n < 100 or math.sqrt(varianza/n) >= d_threshold:
        n += 1
        media_ant = media
        media = media_ant + (func(uniform()) - media_ant) / n
        varianza = varianza * (1 - 1 /(n-1)) + n*(media - media_ant)**2
    return n, media


def fun_ii(x):
    #Como es par, sumo las dos partes.
    def f1(y):
        return (y)**2 * math.exp(-((x)**2))
    return (2*f1(x))


def montecarlo_0_inf(func: Callable, d_threshold):

    def h(u):
        return (1 / (u**2)) * func((1 / u) - 1)

    media = h(uniform())
    varianza, n = 0, 1
    while n < 100 or math.sqrt(varianza/n) >= d_threshold:
        n += 1
        media_ant = media
        media = media_ant + (h(uniform()) - media_ant) / n
        varianza = varianza * (1 - 1 /(n-1)) + n*(media - media_ant)**2
    return n, media

print("Primera integral")
valores, media = montecarlo_0_1(func=fun_i,d_threshold=0.01)
print(f"\tMedia      {media}")
print(f"\tNumero de valores    {valores}")

print("Segunda integral")
valores, media = montecarlo_0_inf(func=fun_ii, d_threshold=0.01)
print(f"\tMedia      {media}")
print(f"\tVarianza   {varianza}")
print(f"\tNumero de valores    {valores}")

Primera integral
	Media      2.0583557776534214
	Numero de valores    50986
Segunda integral
	Media      0.8914933184080615
	Varianza   7.331825779208826
	Numero de valores    12871


Ejercicio 3)

In [ ]:
def montecarlo_ab_intervalo(func, a, b, max_sim, min_sim, threshold):

    media = func(random.random())
    varianza = 0
    n = 1
    intervalo = math.sqrt(varianza / n)

    while n <= min_sim or intervalo > threshold:
        sim = func(a + (b - a) * random.random()) * (b-a) #transformo la x
        media_ant = media
        media = media_ant + (sim - media_ant) / (n + 1)
        varianza_ant = varianza
        varianza = (1 - (1 / n)) * varianza + (n + 1) * (media - media_ant) ** 2
        n += 1
        intervalo = math.sqrt(varianza / n)
    return media, math.sqrt(varianza), intervalo, n


def fun_i(x): #A la transformación la voy a hacer adentro del mc
  return math.sin(x)/x

def fun_ii(x):
  return 3 / (3 + x**4)

#formula de las notas
zalpha_2 = 1.95 # alpha = 0.05
L = 2 * 0.001
d = L / (2 * zalpha_2)

print("test 1")
media1, varianza, intervalo1, numero_valores = montecarlo_ab_intervalo(fun_i, math.pi, 2*math.pi, 100000, 1, d)

print(f"Media   {media1}")
print(f"Desvío    {varianza}")
print(f"Largo del intervalo    {intervalo1}")
print(f"Numero de valores   {numero_valores}")


def montecarlo_0_inf_intervalo(func, max_sim, n_min, threshold):

    media = func(random.random())
    varianza = 0
    n = 1
    intervalo = math.sqrt(varianza / n)

    def h(u):
        return (1 / (u**2)) * func((1 / u) - 1)

    while n <= n_min or intervalo > threshold:
      sim=h(random.random())
      media_ant = media
      media = media_ant + (sim - media_ant) / (n + 1)
      varianza_ant = varianza
      varianza = (1 - (1 / n)) * varianza + (n + 1) * (media - media_ant) ** 2
      n += 1
      intervalo = math.sqrt(varianza / n)
    return media, math.sqrt(varianza), intervalo, n


print("test 2")
media1, varianza, intervalo2, numero_valores = montecarlo_0_inf_intervalo(fun_ii, 100000, 1, d)

print(f"Media   {media1}")
print(f"Desvío    {varianza}")
print(f"Largo del intervalo    {intervalo2}")
print(f"Numero de valores   {numero_valores}")


test 1
Media   -0.4343562334783453
Desvío    0.21058933804609942
Largo del intervalo    0.0005128201676744062
Numero de valores   168633
test 2
Media   1.4624276173022195
Desvío    0.9763276044375259
Largo del intervalo    0.0005128204622747444
Numero de valores   3624603


Ejercicio 4)

In [ ]:
def est_e():
    contador = 0
    suma = 0
    while suma < 1:
        suma += random.random()
        contador += 1
    return contador

print("Inciso a)")

print(f"Valor real {math.e}")
print(f"Valor estimado {sum([est_e() for _ in range(1000)])/1000}")


def varianza_muestral(func, nsim):
  varianza = 0
  n = 1
  mu = func()
  while n <= nsim:
    mu_ant = mu
    mu = mu_ant + (func() - mu_ant) / (n + 1) #f recursiva de mu
    varianza_ant = varianza
    varianza = (1 - (1 / n)) * varianza_ant + (n + 1) * (mu - mu_ant) ** 2 #f recursiva de la varianza
    n += 1
  return varianza

print("Inciso b)")

print(f"Valor estimado de la varianza {varianza_muestral(est_e, 1000)}")


#formula de las notas
zalpha_2 = 1.95 # alpha = 0.05
L = 0.025
d = L / (2 * zalpha_2)


def varianza_con_intervalo(func, threshold):

  varianza = 0
  n = 1
  mu = func()

  while n <= 100 or intervalo >= threshold:
    mu_ant = mu
    mu = mu_ant + (func() - mu_ant) / (n + 1) #f recursiva de mu
    varianza_ant = varianza
    varianza = (1 - (1 / n)) * varianza_ant + (n + 1) * (mu - mu_ant) ** 2 #f recursiva de la varianza
    n += 1
    intervalo = math.sqrt(varianza / n) #Actualizo el ancho del intervalo.
  return varianza, intervalo


print("Inciso c)")
varianza, intervalo = varianza_con_intervalo(est_e, d)
print(f"Valor estimado de la varianza con intervalo {varianza}")
print(f"Largo del intervalo {intervalo}")

Inciso a)
Valor real 2.718281828459045
Valor estimado 2.709
Inciso b)
Valor estimado de la varianza 0.8111528471528413
Inciso c)
Valor estimado de la varianza con intervalo 0.7767246629685696
Largo del intervalo 0.00641014982575977


Ejercicio 5) a)

P(M>n) es igual a 1/n! porque para que M sea el indice del PRIMER valor en cumplir eso, los anteriores no lo tienen que haber cumplido. Eso es una permutación posible de los primeros n números. Como todas las permutaciones tienen la misma probabilidad de ocurrir y hay n! permutaciones, la probabilidad de obtener la permutación donde n es el indice del primer valor no creciente es 1/n!

Inciso b)

In [ ]:
#Cuenta la cantidad de variables que tengo que generar (hasta un limite) para que se rompa una cadena ascendente de generaciones.
def simulacion():

  variable_actual = random.random()
  contador = 1

  while True:
    variable_anterior = variable_actual
    variable_actual = random.random()
    contador += 1
    if variable_actual < variable_anterior:
      break
  return contador


def estimacion(func,n_sim):
  return sum([func() for _ in range(n_sim)])/n_sim


print("Inciso b)")

print(f"Valor real {math.e}")
print(f"Valor estimado {estimacion(simulacion,10000)}")


def varianza_muestral(func, nsim, threshold):
  varianza = 0
  n = 1
  mu = func()
  while n <= nsim:
    mu_ant = mu
    mu = mu_ant + (func() - mu_ant) / (n + 1) #f recursiva de mu
    varianza_ant = varianza
    varianza = (1 - (1 / n)) * varianza_ant + (n + 1) * (mu - mu_ant) ** 2 #f recursiva de la varianza
    n += 1
    if (varianza/n) <= threshold:
      break
  return (varianza/n)

print("Inciso c)")
print(f"Valor de la varianza estimada {varianza_muestral(simulacion, 10000, 0.01)}")

def estimacion_intervalo(func,n_sim, threshold):
  varianza=0
  n=1
  mu=func()
  while n <= n_sim or intervalo >= threshold:
    mu_ant=mu
    mu=mu_ant+(func()-mu_ant)/(n+1)
    varianza_ant=varianza
    varianza=(1-(1/n))*varianza_ant+n*(mu-mu_ant)**2
    n+=1
    intervalo=math.sqrt(varianza/n)
  return mu

zalpha_2 = 1.95 # alpha = 0.05
L = 0.1
d = L / (2 * zalpha_2)

print("Inciso d)")
print(f"Valor estimado {estimacion_intervalo(simulacion, 10000, d)}")

Inciso b)
Valor real 2.718281828459045
Valor estimado 2.7187
Inciso c)
Valor de la varianza estimada 0.0
Inciso d)
Valor estimado 2.7171282871712834


Ejercicio 6)

In [ ]:
def area(n_sim):

    area = 0
    n = 0
    while n <= n_sim:
        x = random.uniform(-1.0, 1.0)
        y = random.uniform(-1.0, 1.0)

        if pow(x, 2) + pow(y, 2) <= 1:
            area += 1

        n += 1
    return 4 * area / n_sim


def f():

    x = random.uniform(-1.0, 1.0)
    y = random.uniform(-1.0, 1.0)

    return pow(x, 2) + pow(y, 2) <= 1


def area_proporcion(sim, threshold, sim_min=100):

    mu = sim()
    n = 1
    S_sq = 0

    while n <= sim_min or math.sqrt(S_sq / n) > threshold:

        mu_ant = mu
        mu = mu_ant + (sim() - mu_ant) / (n + 1)

        S_sq = mu * (1 - mu)
        n += 1

    return mu, n


def area_d_ic(sim, threshold, sim_min=100):

    mu = sim()
    n = 1
    S_sq = 0

    while n <= sim_min or ic > threshold:

        mu_ant = mu

        mu = mu_ant + (sim() - mu_ant) / (n + 1)
        S_sq = mu * (1 - mu)
        n += 1

        ic = math.sqrt(S_sq / n)

    return mu, n, ic


if __name__ == "__main__":

    F = ".5f"
    print(f"valor real  {math.pi:F}")
    print(f"simulacion  {area(10_000):F}")

    print(" ejercicio a ".center(100, "-"))

    p, n = area_proporcion(f, 0.01)

    print(f"proporcion      {p}")
    print(f"simulations     {n}")

    print(" ejercicio b ".center(100, "-"))

    z_alpha_2 = 1.96
    long = 0.1
    d = long / (2 * z_alpha_2)

    mu, n, ic = area_d_ic(sim=f, threshold=d)
    # hacemos 4*mu porque mu es la proporcion de puntos que caen dentro del circulo
    mu *= 4
    print(f"estimacion      {mu}")
    print(f"iter            {n}")
    print(f"ic              ({mu-z_alpha_2*ic}, {mu+z_alpha_2*ic})      dif {1.96*ic}")

valor real  3.141593
simulacion  3.144000
------------------------------------------- ejercicio a --------------------------------------------
proporcion      0.7993769470404992
simulations     1605
------------------------------------------- ejercicio b --------------------------------------------
estimacion      3.309090909090909
iter            220
ic              (3.259139293132168, 3.35904252504965)      dif 0.049951615958741084
